# Staged PINN Solar -- full-batch run on Colab (A100 80 GB) -- NOTEBOOK v2 (gradient accumulated chunk by chunk)

Runs the units of one experiment of protocol v03 (proposal r04) in the **full-batch, float64 regime** of r04 Section 6:
every training device and every interior node enter each L-BFGS / SOAP iteration; chunking is memory only and the
gradient is the exact full-batch gradient (`tests/test_training.py::test_full_batch_chunks_give_the_same_loss_and_gradient_as_one_batch`).

Runtime: **A100 80 GB, High-RAM**. Results land in `Results/<step>/<unit>.json` and are mirrored to Google Drive
after every unit, so a disconnected session resumes where it stopped (complete units are skipped).

Steps, in the protocol's order: `e1` (30 units), `e2` (40), `e5` (10), `e4` (20, optional). Use `UNITS` to pick
a shard (`shard-k-of-n`) or explicit ids such as `e1-A-relative-s0`.


In [ ]:
#@title 1. Runtime check and Google Drive
import subprocess, os, pathlib, json, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before torch is imported
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "no GPU")
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "StagedPINN/Results"  #@param {type:"string"}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MIRROR = pathlib.Path("/content/drive/MyDrive") / DRIVE_DIR
    MIRROR.mkdir(parents=True, exist_ok=True)
else:
    MIRROR = None
print("mirror:", MIRROR)


In [ ]:
#@title 2. Repository (public, pinned to a commit)
REPO = "https://github.com/smindiol/pinn-solar-staged.git"  #@param {type:"string"}
COMMIT = "master"  #@param {type:"string"}
ROOT = pathlib.Path("/content/pinn-solar-staged")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--quiet", REPO, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--quiet", COMMIT], check=True)
print("NOTEBOOK v2 -- expected HEAD ff810f1 or newer"); print("HEAD:", subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([ "pip", "install", "-q", "h5py", "scipy", "optuna", "kagglehub"], check=True)
import sys
sys.path.insert(0, str(ROOT / "src"))
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


In [ ]:
#@title 3. Kaggle credentials and the dPV dataset (sgarizado/dpv-pinn-canonical)
# Put KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (key icon on the left), or upload kaggle.json when asked.
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("credentials: Colab secrets")
except Exception:
    from google.colab import files
    print("upload kaggle.json")
    up = files.upload()
    pathlib.Path("/root/.kaggle").mkdir(exist_ok=True)
    pathlib.Path("/root/.kaggle/kaggle.json").write_bytes(next(iter(up.values())))
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("credentials: kaggle.json")
DATA = pathlib.Path("/content/dpv")
DATA.mkdir(exist_ok=True)
os.environ["DPV_SALIDAS"] = str(DATA)
from Staged_PINN_Solar import data
root = data.ensure_dataset(target=DATA)
print("data root:", root, sorted(p.name for p in pathlib.Path(root).glob("*.h5")))


In [ ]:
#@title 4. Run configuration
STEP = "e1"  #@param ["e1", "e2", "e5", "e4"]
UNITS = ""  #@param {type:"string"}
# "" = every unit of the step; "shard-0-of-3" = one round-robin shard; "e1-A-relative-s0,e1-B-relative-s0" = explicit ids
FULL_BATCH = True  #@param {type:"boolean"}
TRAIN_CHUNK_DEVICES = 16  #@param {type:"integer"}
WALL_BUDGET_HOURS = 22.0  #@param {type:"number"}
SMOKE = False  #@param {type:"boolean"}

from Staged_PINN_Solar import config as C, steps, units as U
cfg = (C.smoke() if SMOKE else C.default()).with_(full_batch=FULL_BATCH, train_chunk_devices=TRAIN_CHUNK_DEVICES,
                                                    wall_budget_seconds=WALL_BUDGET_HOURS * 3600.0)
units_arg = None if not UNITS.strip() else [u.strip() for u in UNITS.split(",") if u.strip()]
every = U.enumerate_units(STEP, cfg)
print(f"{STEP}: {len(every)} units in the step; requested: {units_arg or 'all'}")
RESULTS = ROOT / "Staged-PINN-Solar" / "Results"
os.environ["STAGED_PINN_RESULTS"] = str(RESULTS)
RESULTS.mkdir(parents=True, exist_ok=True)
if MIRROR is not None:     # resume: bring back what an earlier session already finished
    import shutil
    for src in MIRROR.glob(f"{STEP}/*.json"):
        dst = RESULTS / STEP / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
print("results:", RESULTS, "already complete:", len(list((RESULTS / STEP).glob("*.json"))))


In [ ]:
#@title 5. Memory probe (one full-batch gradient, accumulated chunk by chunk) -- size TRAIN_CHUNK_DEVICES
from Staged_PINN_Solar import training
from Staged_PINN_Solar.ansatz import Ansatz
dev = training.device()
prep = training.prepare(config=cfg)
spec = U.parse_unit(every[0])
probe_cfg = cfg.with_(n_blocks=1, iters_per_block=3, max_stages=1)
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
chunks = training.make_batches(prep.train, 0, 0, probe_cfg, dev)
print(f"chunks: {len(chunks)}, devices: {sum(len(c['devices']) for c in chunks)}, nodes in chunk 0: {chunks[0]['x'].shape[0]}")
model = Ansatz(training.build_network(spec, probe_cfg, prep).to(dev), prep.bounds)
obj = training.Objective(model, chunks, 0.0, prep.sigma2, prep.material, prep.contacts, mode=spec.loss)
total, g = obj.value_and_grad({k: 1.0 for k in prep.sigma2}, list(model.parameters()))
peak = torch.cuda.max_memory_allocated() / 2**30
print(f"loss {float(total):.4g}; one full-batch gradient: {time.time()-t0:.1f}s; peak GPU memory {peak:.1f} GiB")
print("peak is ONE chunk's graph: keep it under ~40 GiB (E1's Hessian probe needs about twice a chunk).",
      "Raise TRAIN_CHUNK_DEVICES for speed if far below, lower it if above.")
del model, obj, total, g, chunks
torch.cuda.empty_cache()


In [ ]:
#@title 6. Run the step (resumable; mirrors every finished unit to Drive)
import shutil
def mirror_unit(msg):
    print(msg)
    if MIRROR is not None and ": written " in msg:
        name = msg.split(": written ")[1].split()[0]
        src = RESULTS / STEP / name
        (MIRROR / STEP).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, MIRROR / STEP / name)
run = {"e1": steps.run_e1, "e2": steps.run_e2, "e5": steps.run_e5, "e4": steps.run_e4}[STEP]
t0 = time.time()
done = steps._run_training_step(STEP, units_arg, SMOKE, cfg, None, None, progress=mirror_unit)
print(f"done {len(done)} units in {(time.time()-t0)/3600:.2f} h")


In [ ]:
#@title 7. Quick look at what finished
rows = []
for p in sorted((RESULTS / STEP).glob("*.json")):
    r = json.load(open(p))
    err = [v for v in r.get("per_device_error", {}).values() if v == v]
    med = sorted(err)[len(err)//2] if err else float("nan")
    rows.append((p.stem, r.get("diverged"), f"{med:.2f}", r.get("stages_done"), f"{r.get('wall_seconds', 0)/3600:.2f}h", r.get("lambda_max_seen")))
print(f"{'unit':34} diverged  median_err%  stages  wall   lambda_max")
for row in rows:
    print(f"{row[0]:34} {str(row[1]):8} {row[2]:>10}  {str(row[3]):>6}  {row[4]:>6}  {row[5]}")
